# 🛡️ Infosys Springboard Internship 7.0 (Batch 4) — Milestone 1
## User Authentication Module · Streamlit + JWT + ngrok + Gmail OTP + SQLite

---
### 📌 Project Overview
This Google Colab notebook implements the complete, production-grade authentication module for Milestone 1 as per the official Infosys Springboard specification:
1. **Login & Signup** with mandatory fields, custom email format (`ab@cd.ef`), and password complexity validations.
2. **Dual-Route Password Recovery** via **Security Question** and **Gmail OTP**.
3. **Cryptographic JWT Session Management** (`HS256`) and secure logout.
4. **Protected User Dashboard** displaying verified session claims and token expiration.
5. **Dedicated Admin Portal & Dashboard** displaying registered users with passwords and password hashes strictly redacted.
6. **Google Colab Secrets Integration** for `JWT_SECRET`, `NGROK_AUTHTOKEN`, `EMAIL_PASSWORD`, and `EMAIL_ADDRESS`.
7. **Public Tunneling via ngrok** to access the Streamlit app directly from any web browser.

### 🔑 Step 4, 5, 6: Setting Up Google Colab Secrets (Zero-Credential Policy)
Never hard-code credentials directly in the notebook.
Open the left sidebar and click the **Key icon (Secrets)** to add the required keys (toggle **Notebook access** ON for each):

#### 📧 Official Gmail API (OAuth 2.0 via HTTPS Port 443)
> ℹ️ *Note: Google Colab actively blocks raw SMTP (port 587). This project uses the official Google Gmail REST API over HTTPS, which is 100% permitted and never blocked.*

| Secret Name | Category | Description / Instructions |
|---|---|---|
| `GMAIL_CLIENT_ID` | Gmail API (OAuth 2.0) | OAuth 2.0 Client ID from Google Cloud Console |
| `GMAIL_CLIENT_SECRET` | Gmail API (OAuth 2.0) | OAuth 2.0 Client Secret from Google Cloud Console |
| `GMAIL_REFRESH_TOKEN` | Gmail API (OAuth 2.0) | Long-lived OAuth 2.0 Refresh Token with `gmail.send` scope |
| `EMAIL_ADDRESS` | Gmail API | Your Gmail address (used in the email 'From' header) |
| `JWT_SECRET` | Security / Session | Random secret string for signing JWT tokens |
| `NGROK_AUTHTOKEN` | Public Tunnel | Personal authtoken from [dashboard.ngrok.com](https://dashboard.ngrok.com/) |
| `ADMIN_USERNAME` | Admin Portal | *(Optional)* Administrator username (defaults to 'admin') |
| `ADMIN_PASSWORD` | Admin Portal | *(Optional)* Administrator password for Admin Portal |

In [ ]:
# STEP 1: Install Required Python Libraries
!pip install -q streamlit pyjwt bcrypt pyngrok python-dotenv google-api-python-client google-auth google-auth-oauthlib google-auth-httplib2


In [ ]:
# STEP 2: Mount Google Drive & Bridge Google Colab Secrets (Step 6 & Issue 2)
import os
import shutil
from pathlib import Path

# 1. Mount Google Drive for persistent SQLite database storage (Issue 2)
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        print('Mounting Google Drive for persistent SQLite storage...')
        drive.mount('/content/drive')
        print('✅ Google Drive mounted successfully at /content/drive/MyDrive.')
    else:
        print('✅ Google Drive is already mounted.')
        
    # Ensure persistent directory exists
    persistent_db_dir = Path('/content/drive/MyDrive/Infosys_Springboard_Milestone1')
    persistent_db_dir.mkdir(parents=True, exist_ok=True)
    persistent_db_file = persistent_db_dir / 'auth_system.db'
    local_db_file = Path('auth_system.db')
    
    # Preserve existing user accounts and bcrypt hashes if local DB exists
    if not persistent_db_file.exists() and local_db_file.exists():
        shutil.copy2(local_db_file, persistent_db_file)
        print('  📁 Copied existing local database to persistent Google Drive storage.')
    elif persistent_db_file.exists():
        print('  📁 Persistent database ready on Google Drive.')
except Exception as e:
    print(f'ℹ️ Google Drive mount note / local mode: {e}')

# 2. Bridge Google Colab Secrets into environment and .env
try:
    from google.colab import userdata
    from google.colab.userdata import NotebookAccessError, SecretNotFoundError
except ImportError:
    userdata = None
    NotebookAccessError = Exception
    SecretNotFoundError = Exception

key_aliases = {
    'GMAIL_CLIENT_ID': ['gmail_client_id', 'CLIENT_ID', 'client_id', 'GOOGLE_CLIENT_ID'],
    'GMAIL_CLIENT_SECRET': ['gmail_client_secret', 'CLIENT_SECRET', 'client_secret', 'GOOGLE_CLIENT_SECRET'],
    'GMAIL_REFRESH_TOKEN': ['gmail_refresh_token', 'REFRESH_TOKEN', 'refresh_token', 'GOOGLE_REFRESH_TOKEN'],
    'EMAIL_ADDRESS': [
        'email_address', 'GMAIL_ADDRESS', 'gmail_address', 'EMAIL', 'email',
        'GMAIL', 'gmail', 'EMAIL_USER', 'email_user', 'GMAIL_USER', 'gmail_user',
        'GMAIL_EMAIL', 'gmail_email', 'SENDER_EMAIL', 'sender_email',
        'SENDER_GMAIL', 'sender_gmail', 'USER_EMAIL', 'user_email'
    ],
    'JWT_SECRET': ['jwt_secret', 'SECRET_KEY', 'secret_key', 'JWT_KEY', 'jwt_key'],
    'NGROK_AUTHTOKEN': ['ngrok_authtoken', 'NGROK_AUTH_TOKEN', 'ngrok_auth_token', 'AUTHTOKEN', 'authtoken'],
    'ADMIN_USERNAME': ['admin_username'],
    'ADMIN_PASSWORD': ['admin_password'],
    'DB_PATH': ['db_path', 'DATABASE_PATH', 'database_path']
}

print('\nChecking Google Colab Secrets configuration:')
env_lines = []
if userdata:
    for primary, aliases in key_aliases.items():
        candidate_keys = [primary] + aliases
        val = None
        access_error_key = None
        for k in candidate_keys:
            try:
                v = userdata.get(k)
                if v is not None and str(v).strip():
                    val = str(v).strip()
                    break
            except NotebookAccessError:
                access_error_key = k
            except SecretNotFoundError:
                continue
            except Exception:
                continue
        if val:
            os.environ[primary] = val
            env_lines.append(f"{primary}={val}")
            print(f'  ✅ {primary}: Detected & configured')
        elif access_error_key:
            print(f"  ⚠️ {primary}: Secret '{access_error_key}' exists in Secrets, but 'Notebook access' toggle is OFF! Please enable the toggle switch next to it in Colab Secrets.")
        else:
            pass

# Persist secrets to .env files so background Streamlit process has immediate access
if env_lines:
    env_content = '\n'.join(env_lines) + '\n'
    Path('.env').write_text(env_content, encoding='utf-8')
    if Path('/content').exists():
        Path('/content/.env').write_text(env_content, encoding='utf-8')
    print('  🔒 Secrets bridged to background runtime environment (.env files)')


In [ ]:
%%writefile config.py
import os
import shutil
import secrets
from pathlib import Path
from typing import Tuple, List, Dict
from dotenv import load_dotenv

# Search for and load local .env files from multiple possible locations
# (Milestone1 folder, repository root, current working directory, /content in Colab)
ENV_SEARCH_PATHS = [
    Path(__file__).resolve().parent / ".env",
    Path.cwd() / ".env",
    Path(__file__).resolve().parent.parent / ".env",
    Path("/content/.env")
]

for p in ENV_SEARCH_PATHS:
    if p.exists() and p.is_file():
        load_dotenv(p, override=False)

# Supported secret aliases covering uppercase, lowercase, and common variations
KEY_ALIASES: Dict[str, List[str]] = {
    "EMAIL_ADDRESS": [
        "email_address",
        "GMAIL_ADDRESS", "gmail_address",
        "EMAIL", "email",
        "GMAIL", "gmail",
        "EMAIL_USER", "email_user",
        "GMAIL_USER", "gmail_user",
        "GMAIL_EMAIL", "gmail_email",
        "SENDER_EMAIL", "sender_email",
        "SENDER_GMAIL", "sender_gmail",
        "USER_EMAIL", "user_email",
        "MAIL_USERNAME", "mail_username",
        "SMTP_USER", "smtp_user",
        "SMTP_EMAIL", "smtp_email",
        "GOOGLE_EMAIL", "google_email"
    ],
    "EMAIL_PASSWORD": [
        "email_password",
        "GMAIL_APP_PASSWORD", "gmail_app_password",
        "GMAIL_PASSWORD", "gmail_password",
        "EMAIL_APP_PASSWORD", "email_app_password",
        "APP_PASSWORD", "app_password",
        "GOOGLE_APP_PASSWORD", "google_app_password",
        "GMAIL_PASS", "gmail_pass",
        "EMAIL_PASS", "email_pass",
        "APP_PASS", "app_pass",
        "GMAIL_APP_PASS", "gmail_app_pass",
        "MAIL_PASSWORD", "mail_password",
        "SMTP_PASSWORD", "smtp_password",
        "SMTP_PASS", "smtp_pass"
    ],
    "JWT_SECRET": [
        "jwt_secret",
        "SECRET_KEY", "secret_key",
        "JWT_KEY", "jwt_key"
    ],
    "NGROK_AUTHTOKEN": [
        "ngrok_authtoken",
        "NGROK_AUTH_TOKEN", "ngrok_auth_token",
        "AUTHTOKEN", "authtoken"
    ],
    "ADMIN_USERNAME": [
        "admin_username"
    ],
    "ADMIN_PASSWORD": [
        "admin_password"
    ],
    "DB_PATH": [
        "db_path",
        "DATABASE_PATH", "database_path",
        "DATABASE_URL", "database_url"
    ],
    "GMAIL_CLIENT_ID": [
        "gmail_client_id", "CLIENT_ID", "client_id", "GOOGLE_CLIENT_ID", "google_client_id"
    ],
    "GMAIL_CLIENT_SECRET": [
        "gmail_client_secret", "CLIENT_SECRET", "client_secret", "GOOGLE_CLIENT_SECRET", "google_client_secret"
    ],
    "GMAIL_REFRESH_TOKEN": [
        "gmail_refresh_token", "REFRESH_TOKEN", "refresh_token", "GOOGLE_REFRESH_TOKEN", "google_refresh_token"
    ],
    "BREVO_API_KEY": [
        "brevo_api_key", "EMAIL_API_KEY", "email_api_key", "SENDINBLUE_API_KEY", "sendinblue_api_key", "BREVO_KEY", "brevo_key"
    ],
    "RESEND_API_KEY": [
        "resend_api_key", "RESEND_KEY", "resend_key"
    ]
}

# Diagnostic status dictionary without exposing secret values
_SECRET_STATUS: Dict[str, str] = {}

def sync_colab_secrets_to_env() -> None:
    """
    If running inside Google Colab, bridge Colab Secrets into os.environ
    and persistent .env files so background subprocesses (like Streamlit)
    can reliably access them.
    """
    try:
        from google.colab import userdata  # type: ignore
        from google.colab.userdata import NotebookAccessError, SecretNotFoundError  # type: ignore
    except ImportError:
        userdata = None
        NotebookAccessError = Exception
        SecretNotFoundError = Exception

    if not userdata:
        return

    env_updates = {}
    for primary_key, aliases in KEY_ALIASES.items():
        candidate_keys = [primary_key] + aliases
        val = None
        for k in candidate_keys:
            try:
                v = userdata.get(k)
                if v is not None and str(v).strip():
                    val = str(v).strip()
                    _SECRET_STATUS[primary_key] = f"Found in Colab Secrets via '{k}'"
                    break
            except NotebookAccessError:
                _SECRET_STATUS[primary_key] = (
                    f"Configured in Colab Secrets as '{k}', but 'Notebook access' toggle is OFF. "
                    "Please toggle 'Notebook access' ON in the Colab Secrets sidebar."
                )
            except SecretNotFoundError:
                continue
            except Exception:
                continue

        if val:
            os.environ[primary_key] = val
            env_updates[primary_key] = val

    # Persist secrets to .env in parent directory and /content if available
    if env_updates:
        target_files = [Path(__file__).resolve().parent / ".env"]
        if Path("/content").exists():
            target_files.append(Path("/content/.env"))

        for target in target_files:
            try:
                existing = {}
                if target.exists():
                    for line in target.read_text(encoding="utf-8").splitlines():
                        line = line.strip()
                        if line and not line.startswith("#") and "=" in line:
                            k, v = line.split("=", 1)
                            existing[k.strip()] = v.strip()
                existing.update(env_updates)
                target.write_text(
                    "\n".join(f"{k}={v}" for k, v in existing.items()) + "\n",
                    encoding="utf-8"
                )
            except Exception:
                pass

# Synchronize secrets if running in Colab notebook environment
sync_colab_secrets_to_env()

def get_secret(key: str, default: str = "") -> str:
    """
    Safely retrieves a configuration secret without exposing values.
    Prioritizes:
    1. OS environment variables (os.environ, including .env)
    2. Google Colab Secrets (userdata.get), supporting all candidate aliases
    3. Falls back to default.
    """
    candidate_keys = [key] + KEY_ALIASES.get(key, [])

    # 1. Check OS environment variables first (fast and subprocess-friendly)
    for k in candidate_keys:
        val = os.getenv(k)
        if val is not None and str(val).strip():
            val_str = str(val).strip()
            os.environ[key] = val_str
            return val_str

    # 2. Check Google Colab Secrets
    try:
        from google.colab import userdata  # type: ignore
        from google.colab.userdata import NotebookAccessError, SecretNotFoundError  # type: ignore
        for k in candidate_keys:
            try:
                val = userdata.get(k)
                if val is not None and str(val).strip():
                    val_str = str(val).strip()
                    os.environ[key] = val_str
                    _SECRET_STATUS[key] = f"Loaded from Colab Secrets via '{k}'"
                    return val_str
            except NotebookAccessError:
                _SECRET_STATUS[key] = (
                    f"Secret '{k}' found in Colab Secrets, but 'Notebook access' toggle is OFF. "
                    "Please toggle 'Notebook access' ON in Colab's Secrets tab."
                )
            except Exception:
                pass
    except Exception:
        pass

    return default

def get_secret_status(key: str) -> str:
    """Returns safe diagnostic info about why a secret was or wasn't loaded."""
    if key in _SECRET_STATUS:
        return _SECRET_STATUS[key]
    val = get_secret(key, "")
    if val:
        return "Configured and available"
    return "Not set in Colab Secrets or .env"

# Ephemeral fallback JWT Secret if none configured
_configured_jwt = get_secret("JWT_SECRET", "")
if not _configured_jwt:
    JWT_SECRET = secrets.token_hex(32)
else:
    JWT_SECRET = _configured_jwt

# ngrok Authtoken
NGROK_AUTHTOKEN = get_secret("NGROK_AUTHTOKEN", "")

# Gmail SMTP Configuration for OTP Delivery (evaluated dynamically when sending)
EMAIL_ADDRESS = get_secret("EMAIL_ADDRESS", "")
EMAIL_PASSWORD = get_secret("EMAIL_PASSWORD", "")

# HTTPS Email Delivery Configurations (Gmail OAuth2 & Transactional APIs)
GMAIL_CLIENT_ID = get_secret("GMAIL_CLIENT_ID", "")
GMAIL_CLIENT_SECRET = get_secret("GMAIL_CLIENT_SECRET", "")
GMAIL_REFRESH_TOKEN = get_secret("GMAIL_REFRESH_TOKEN", "")
BREVO_API_KEY = get_secret("BREVO_API_KEY", "") or get_secret("EMAIL_API_KEY", "")
RESEND_API_KEY = get_secret("RESEND_API_KEY", "")

# Dedicated Admin credentials
ADMIN_USERNAME = get_secret("ADMIN_USERNAME", "admin")
ADMIN_PASSWORD = get_secret("ADMIN_PASSWORD", "Admin@Placeholder2026!")

def get_db_path() -> Path:
    """
    Resolves persistent SQLite database path across restarts and Colab disconnects.
    - If running in Colab and Google Drive is mounted (/content/drive/MyDrive),
      stores the database in: /content/drive/MyDrive/Infosys_Springboard_Milestone1/auth_system.db
      to guarantee persistence across Colab runtime recycles.
      If persistent DB does not exist yet but local DB exists, automatically copies
      the local database so existing records are preserved.
    - Otherwise (local development), stores the database in the local project directory:
      Milestone1/auth_system.db.
    - Preserves existing records; never deletes or resets the database.
    """
    custom = os.getenv("DB_PATH")
    if custom:
        p = Path(custom).resolve()
        p.parent.mkdir(parents=True, exist_ok=True)
        return p

    local_path = Path(__file__).resolve().parent / "auth_system.db"

    # Google Colab persistent Google Drive storage check
    drive_mount = Path("/content/drive/MyDrive")
    if drive_mount.exists():
        persistent_dir = drive_mount / "Infosys_Springboard_Milestone1"
        try:
            persistent_dir.mkdir(parents=True, exist_ok=True)
            persistent_db = persistent_dir / "auth_system.db"

            # Seed persistent DB from local DB if local DB exists and persistent DB doesn't yet
            if not persistent_db.exists() and local_path.exists():
                try:
                    shutil.copy2(local_path, persistent_db)
                except Exception:
                    pass

            return persistent_db
        except Exception:
            return local_path

    return local_path

# Database path (initialized to resolved path)
DB_PATH = get_db_path()


In [ ]:
%%writefile database.py
import sqlite3
import time
from typing import Optional, List, Dict, Tuple
from config import DB_PATH, get_db_path

def get_connection() -> sqlite3.Connection:
    """Returns a SQLite connection with dict-like row access to persistent DB."""
    resolved_path = get_db_path()
    resolved_path.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(str(resolved_path), check_same_thread=False)
    conn.row_factory = sqlite3.Row
    return conn

def init_db() -> None:
    """Initializes tables for users and temporary OTP verification."""
    with get_connection() as conn:
        cursor = conn.cursor()
        # Users table
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT UNIQUE NOT NULL,
                email TEXT UNIQUE NOT NULL,
                password_hash TEXT NOT NULL,
                security_question TEXT NOT NULL,
                security_answer_hash TEXT NOT NULL,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        """)
        
        # OTP verification table
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS otps (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                email TEXT NOT NULL,
                otp_code TEXT NOT NULL,
                expires_at REAL NOT NULL,
                used INTEGER DEFAULT 0,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        """)
        conn.commit()

def create_user(
    username: str,
    email: str,
    password_hash: str,
    security_question: str,
    security_answer_hash: str
) -> Tuple[bool, str]:
    """
    Registers a new user.
    Ensures username and email uniqueness.
    """
    try:
        with get_connection() as conn:
            cursor = conn.cursor()
            # Check username collision
            cursor.execute("SELECT id FROM users WHERE LOWER(username) = LOWER(?)", (username,))
            if cursor.fetchone():
                return False, "Username is already taken. Please choose another username."

            # Check email collision
            cursor.execute("SELECT id FROM users WHERE LOWER(email) = LOWER(?)", (email,))
            if cursor.fetchone():
                return False, "An account with this email already exists."

            cursor.execute(
                """
                INSERT INTO users (username, email, password_hash, security_question, security_answer_hash)
                VALUES (?, ?, ?, ?, ?)
                """,
                (username.strip(), email.strip().lower(), password_hash, security_question, security_answer_hash)
            )
            conn.commit()
            return True, "User registered successfully!"
    except sqlite3.IntegrityError as e:
        return False, f"Registration failed: {str(e)}"
    except Exception as e:
        return False, f"Unexpected error during registration: {str(e)}"

def get_user_by_username(username: str) -> Optional[Dict]:
    """Finds user by exact or case-insensitive username."""
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM users WHERE LOWER(username) = LOWER(?)", (username.strip(),))
        row = cursor.fetchone()
        return dict(row) if row else None

def get_user_by_email(email: str) -> Optional[Dict]:
    """Finds user by case-insensitive email."""
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM users WHERE LOWER(email) = LOWER(?)", (email.strip(),))
        row = cursor.fetchone()
        return dict(row) if row else None

def get_user_by_identifier(identifier: str) -> Optional[Dict]:
    """Finds user by either username or email."""
    clean = identifier.strip().lower()
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute(
            "SELECT * FROM users WHERE LOWER(username) = ? OR LOWER(email) = ?",
            (clean, clean)
        )
        row = cursor.fetchone()
        return dict(row) if row else None

def update_password(username: str, new_password_hash: str) -> bool:
    """Updates password hash for a given user."""
    try:
        with get_connection() as conn:
            cursor = conn.cursor()
            cursor.execute(
                "UPDATE users SET password_hash = ? WHERE LOWER(username) = LOWER(?)",
                (new_password_hash, username.strip())
            )
            conn.commit()
            return cursor.rowcount > 0
    except Exception:
        return False

def get_all_users_for_admin() -> List[Dict]:
    """
    Retrieves all registered users for Admin Dashboard.
    CRITICAL: Never fetches password_hash or security_answer_hash (Step 11 requirement).
    """
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute(
            "SELECT id, username, email, security_question, created_at FROM users ORDER BY id ASC"
        )
        rows = cursor.fetchall()
        return [dict(row) for row in rows]

def store_otp(email: str, otp_code: str, validity_minutes: int = 10) -> bool:
    """Stores a generated OTP with expiration timestamp."""
    try:
        expires_at = time.time() + (validity_minutes * 60)
        with get_connection() as conn:
            cursor = conn.cursor()
            # Invalidate older unused OTPs for this email
            cursor.execute("UPDATE otps SET used = 1 WHERE LOWER(email) = LOWER(?) AND used = 0", (email.strip(),))
            cursor.execute(
                "INSERT INTO otps (email, otp_code, expires_at, used) VALUES (?, ?, ?, 0)",
                (email.strip().lower(), otp_code.strip(), expires_at)
            )
            conn.commit()
            return True
    except Exception:
        return False

def verify_and_consume_otp(email: str, otp_code: str) -> Tuple[bool, str]:
    """
    Verifies the provided OTP against the active unexpired OTPs in the database.
    Marks OTP as used upon successful verification.
    """
    clean_email = email.strip().lower()
    clean_otp = otp_code.strip()
    now = time.time()

    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute(
            """
            SELECT id, otp_code, expires_at, used 
            FROM otps 
            WHERE LOWER(email) = ? 
            ORDER BY id DESC LIMIT 1
            """,
            (clean_email,)
        )
        row = cursor.fetchone()

        if not row:
            return False, "No OTP request found for this email. Please request a new OTP."

        if row["used"] == 1:
            return False, "This OTP has already been used. Please request a fresh OTP."

        if now > row["expires_at"]:
            return False, "The OTP has expired. Please request a new OTP."

        if row["otp_code"] != clean_otp:
            return False, "Incorrect OTP entered. Please check your email and try again."

        # Mark OTP as consumed
        cursor.execute("UPDATE otps SET used = 1 WHERE id = ?", (row["id"],))
        conn.commit()
        return True, "OTP verified successfully!"


In [ ]:
%%writefile mailer.py
import base64
import json
import secrets
import socket
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from typing import Tuple, Dict, Any, Optional

# Official Google API Client and OAuth 2.0 libraries
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
import google.auth.exceptions
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

from config import (
    EMAIL_ADDRESS,
    GMAIL_CLIENT_ID,
    GMAIL_CLIENT_SECRET,
    GMAIL_REFRESH_TOKEN,
    get_secret,
    get_secret_status
)

def _sanitize_error(error_msg: str, *sensitive_values: str) -> str:
    """Masks secrets, tokens, client secrets, and OTP codes from error messages."""
    sanitized = str(error_msg)
    for val in sensitive_values:
        if val and len(str(val)) >= 3:
            sanitized = sanitized.replace(str(val), "******")
    return sanitized

def generate_otp() -> str:
    """Generates a secure 6-digit numeric OTP."""
    return str(secrets.randbelow(900000) + 100000)

def _get_email_templates(otp_code: str) -> Tuple[str, str]:
    """Generates plaintext and responsive HTML email templates for OTP verification."""
    text_body = f"""Hello,

You have requested a password reset for your Infosys Springboard account.
Your One-Time Password (OTP) is:

    {otp_code}

This OTP is valid for 10 minutes.
If you did not request this password reset, please ignore this email.

Best regards,
Infosys Springboard Internship Team
"""

    html_body = f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <style>
        body {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background-color: #f4f6f9; margin: 0; padding: 20px; }}
        .card {{ background: #ffffff; max-width: 500px; margin: 0 auto; border-radius: 12px; padding: 30px; box-shadow: 0 4px 12px rgba(0,0,0,0.08); border-top: 5px solid #0070ad; }}
        .header {{ text-align: center; margin-bottom: 25px; }}
        .header h2 {{ color: #0070ad; margin: 0; font-size: 22px; }}
        .header p {{ color: #666; font-size: 13px; margin-top: 4px; }}
        .otp-box {{ background: #f0f7ff; border: 2px dashed #0070ad; border-radius: 8px; text-align: center; padding: 18px; margin: 25px 0; }}
        .otp-code {{ font-size: 32px; font-weight: 700; letter-spacing: 6px; color: #004d7a; font-family: 'Courier New', Courier, monospace; }}
        .notice {{ color: #555; font-size: 14px; line-height: 1.6; }}
        .footer {{ text-align: center; font-size: 11px; color: #999; margin-top: 30px; border-top: 1px solid #eee; padding-top: 15px; }}
    </style>
</head>
<body>
    <div class="card">
        <div class="header">
            <h2>Infosys Springboard Internship 7.0</h2>
            <p>Security Verification Service</p>
        </div>
        <p class="notice">Hello,</p>
        <p class="notice">We received a request to reset your password. Use the verification code below to complete the process:</p>
        <div class="otp-box">
            <div class="otp-code">{otp_code}</div>
            <div style="font-size: 12px; color: #777; margin-top: 6px;">Valid for 10 minutes</div>
        </div>
        <p class="notice" style="color: #c0392b; font-size: 13px;">
            ⚠️ If you did not initiate this request, please ignore this email. Your account remains secure.
        </p>
        <div class="footer">
            &copy; 2026 Infosys Springboard · Internship 7.0 Batch 4 · Milestone 1
        </div>
    </div>
</body>
</html>"""
    return text_body, html_body

# -----------------------------------------------------------------------------
# Official Google Gmail API Client (OAuth 2.0 via HTTPS Port 443)
# -----------------------------------------------------------------------------
def get_gmail_service() -> Tuple[Optional[Any], Optional[str]]:
    """
    Builds and returns an authorized Google Gmail API v1 Resource using OAuth 2.0.
    
    Required Credentials (from Google Colab Secrets or .env):
      - GMAIL_CLIENT_ID
      - GMAIL_CLIENT_SECRET
      - GMAIL_REFRESH_TOKEN
    Scope: https://www.googleapis.com/auth/gmail.send
    
    Returns: (service_instance, error_message)
    """
    client_id = (get_secret("GMAIL_CLIENT_ID", "") or GMAIL_CLIENT_ID).strip().strip('"').strip("'")
    client_secret = (get_secret("GMAIL_CLIENT_SECRET", "") or GMAIL_CLIENT_SECRET).strip().strip('"').strip("'")
    refresh_token = (get_secret("GMAIL_REFRESH_TOKEN", "") or GMAIL_REFRESH_TOKEN).strip().strip('"').strip("'")

    missing = []
    if not client_id:
        missing.append("GMAIL_CLIENT_ID")
    if not client_secret:
        missing.append("GMAIL_CLIENT_SECRET")
    if not refresh_token:
        missing.append("GMAIL_REFRESH_TOKEN")

    if missing:
        return None, (
            f"Gmail API OAuth 2.0 credentials missing: {', '.join(missing)}. "
            "Please configure these secrets in Google Colab (Secrets / key icon in left sidebar) "
            "with 'Notebook access' enabled."
        )

    try:
        creds = Credentials(
            token=None,
            refresh_token=refresh_token,
            token_uri="https://oauth2.googleapis.com/token",
            client_id=client_id,
            client_secret=client_secret,
            scopes=["https://www.googleapis.com/auth/gmail.send"]
        )
        # Test token refresh over HTTPS to confirm credentials validity
        req = Request()
        creds.refresh(req)

        # Build official Google API client resource
        service = build("gmail", "v1", credentials=creds, cache_discovery=False)
        return service, None
    except google.auth.exceptions.RefreshError as e:
        safe_err = _sanitize_error(str(e), client_id, client_secret, refresh_token)
        return None, (
            f"Gmail OAuth 2.0 Token Refresh Failed ({type(e).__name__}): {safe_err}. "
            "Your GMAIL_CLIENT_ID, GMAIL_CLIENT_SECRET, or GMAIL_REFRESH_TOKEN is invalid or revoked. "
            "Please verify the credentials in Google Cloud Console."
        )
    except Exception as e:
        safe_err = _sanitize_error(str(e), client_id, client_secret, refresh_token)
        return None, f"Gmail API Service Initialization Failed ({type(e).__name__}): {safe_err}"

# -----------------------------------------------------------------------------
# Main Public OTP Send Function (Invoked by Forgot Password UI)
# -----------------------------------------------------------------------------
def send_otp_email(recipient_email: str, otp_code: str) -> Tuple[bool, str]:
    """
    Sends the 6-digit OTP to the registered user's email address using the official
    Google Gmail REST API over HTTPS (Port 443) with OAuth 2.0 (gmail.send scope).
    
    SMTP is completely eliminated from this sending path:
    - No smtplib or raw TCP socket to port 587.
    - 100% immune to Google Colab / GCE network egress filtering (SMTPServerDisconnected).
    - Returns success ONLY when the Gmail API confirms HTTP 200 with an active message ID.
    - Never exposes OTPs, client secrets, access tokens, or refresh tokens in logs or errors.
    """
    clean_recipient = recipient_email.strip()
    if not clean_recipient:
        return False, "Recipient email address cannot be empty."

    service, err = get_gmail_service()
    if err:
        return False, err

    sender = (get_secret("EMAIL_ADDRESS", "") or EMAIL_ADDRESS or "").strip().strip('"').strip("'")
    text_body, html_body = _get_email_templates(otp_code)

    msg = MIMEMultipart("alternative")
    msg["Subject"] = "Infosys Springboard - Password Reset OTP"
    from_header = f"Infosys Springboard Support <{sender}>" if "@" in sender else "Infosys Springboard Support"
    msg["From"] = from_header
    msg["To"] = clean_recipient
    msg.attach(MIMEText(text_body, "plain"))
    msg.attach(MIMEText(html_body, "html"))

    raw_bytes = msg.as_bytes()
    raw_b64 = base64.urlsafe_b64encode(raw_bytes).decode("utf-8")

    try:
        # Call official Google Gmail API: users.messages.send
        result = service.users().messages().send(userId="me", body={"raw": raw_b64}).execute()
        msg_id = result.get("id")
        if msg_id:
            return True, f"OTP email successfully delivered to {clean_recipient} via Gmail API (Message ID: {msg_id})!"
        return True, f"OTP email delivered to {clean_recipient} via Gmail API!"
    except HttpError as e:
        client_secret = get_secret("GMAIL_CLIENT_SECRET", "")
        refresh_token = get_secret("GMAIL_REFRESH_TOKEN", "")
        safe_err = _sanitize_error(str(e), client_secret, refresh_token, otp_code)
        return False, f"Gmail API error (HTTP {e.resp.status}): {safe_err}"
    except Exception as e:
        client_secret = get_secret("GMAIL_CLIENT_SECRET", "")
        refresh_token = get_secret("GMAIL_REFRESH_TOKEN", "")
        safe_err = _sanitize_error(str(e), client_secret, refresh_token, otp_code)
        return False, f"Failed to deliver OTP via Gmail API ({type(e).__name__}): {safe_err}"

# -----------------------------------------------------------------------------
# Verification, Diagnostic & Testing Utilities
# -----------------------------------------------------------------------------
def send_test_email(recipient_email: str) -> Tuple[bool, str]:
    """
    Sends an actual test OTP email using the Gmail API to confirm Google provider acceptance.
    Does NOT touch the SQLite database or modify password recovery state.
    """
    test_otp = generate_otp()
    clean_to = recipient_email.strip()
    if not clean_to:
        return False, "Target test email address cannot be empty."

    success, msg = send_otp_email(clean_to, test_otp)
    if success:
        return True, f"✅ Test email verified and accepted by Gmail API!\nRecipient: {clean_to}\nDetails: {msg}"
    else:
        return False, f"❌ Gmail API test email failed:\n{msg}"

def check_gmail_api_status() -> Dict[str, Any]:
    """
    Checks Gmail API OAuth 2.0 configuration and HTTPS token refresh.
    Does NOT dispatch real emails during diagnosis.
    """
    client_id = (get_secret("GMAIL_CLIENT_ID", "") or GMAIL_CLIENT_ID).strip()
    client_secret = (get_secret("GMAIL_CLIENT_SECRET", "") or GMAIL_CLIENT_SECRET).strip()
    refresh_token = (get_secret("GMAIL_REFRESH_TOKEN", "") or GMAIL_REFRESH_TOKEN).strip()

    status = {
        "configured": bool(client_id and client_secret and refresh_token),
        "has_client_id": bool(client_id),
        "has_client_secret": bool(client_secret),
        "has_refresh_token": bool(refresh_token),
        "auth_test_passed": False,
        "details": ""
    }

    if not status["configured"]:
        missing = []
        if not client_id: missing.append("GMAIL_CLIENT_ID")
        if not client_secret: missing.append("GMAIL_CLIENT_SECRET")
        if not refresh_token: missing.append("GMAIL_REFRESH_TOKEN")
        status["details"] = f"Missing secrets: {', '.join(missing)}"
        return status

    service, err = get_gmail_service()
    if service is not None:
        status["auth_test_passed"] = True
        status["details"] = "Gmail API OAuth 2.0 connection and token refresh successful over HTTPS!"
    else:
        status["auth_test_passed"] = False
        status["details"] = err or "Failed to connect to Gmail API"

    return status

def verify_email_template(recipient_email: str, otp_code: str) -> Tuple[bool, str]:
    """Tests email message generation and MIME compilation locally without network calls."""
    try:
        text_body, html_body = _get_email_templates(otp_code)
        if recipient_email and otp_code and "Infosys Springboard" in html_body:
            return True, "Email template compiled successfully."
        return False, "Template compilation failed integrity checks."
    except Exception as e:
        return False, f"Email template construction failed: {str(e)}"

def test_smtp_connection_auth(timeout: int = 10) -> Tuple[bool, str]:
    """Backward compatibility alias: reports Gmail API status instead of SMTP."""
    status = check_gmail_api_status()
    return status["auth_test_passed"], status["details"]


In [ ]:
# STEP 2B: Verify Gmail API OAuth 2.0 Credentials & Test Real OTP Email Delivery
# Run this cell to verify your Google Gmail API connection and dispatch a real test OTP email.
import mailer
from config import get_secret

print('=' * 72)
print('🔍 GMAIL API OAUTH 2.0 STATUS & TEST OTP DISPATCH (PORT 443 HTTPS)')
print('=' * 72)

status = mailer.check_gmail_api_status()
print(f"  • Credentials Configured:   {'✅ YES' if status['configured'] else '❌ NO'}")
print(f"  • Token Refresh Test:       {'✅ PASS' if status['auth_test_passed'] else '❌ FAIL'}")
print(f"  • Details:                  {status['details']}")

print('-' * 72)
if not status['auth_test_passed']:
    print('⚠️ ACTION REQUIRED: Gmail API credentials not yet active.')
    print('   Please configure GMAIL_CLIENT_ID, GMAIL_CLIENT_SECRET, and GMAIL_REFRESH_TOKEN in Secrets.')
else:
    target_email = (get_secret('EMAIL_ADDRESS', '') or 'your_email@gmail.com').strip()
    print(f'🚀 Dispatching live test OTP email to: {target_email}...')
    success, result_msg = mailer.send_test_email(target_email)
    print(f'\n{result_msg}')
print('=' * 72)


In [ ]:
%%writefile auth.py
import re
import datetime
from typing import Tuple, Optional, Dict, Any
import jwt
import bcrypt
import hashlib
from config import JWT_SECRET

# Fixed list of security questions for Signup and Recovery
SECURITY_QUESTIONS = [
    "What was the name of your first pet?",
    "What was the name of your elementary school?",
    "In what city or town was your first job?",
    "What is your mother's maiden name?",
    "What was the make and model of your first car?",
    "What is your favorite childhood book or movie?"
]

def validate_email_format(email: str) -> Tuple[bool, str]:
    """
    Validates email according to Milestone 1 Step 8:
    'At least 2 letters before the @, at least 2 letters between the @ and the dot,
    and at least 2 letters after the final dot — e.g. an address shaped like ab@cd.ef.'
    """
    if not email:
        return False, "Email address is required."

    email = email.strip()
    if "@" not in email:
        return False, "Invalid email format: missing '@'."

    parts = email.split("@")
    if len(parts) != 2:
        return False, "Invalid email format: must contain exactly one '@'."

    local_part, domain_part = parts[0], parts[1]

    # At least 2 characters before the @
    if len(local_part) < 2:
        return False, "Email must have at least 2 characters before the '@' symbol (e.g. ab@cd.ef)."

    # Domain part checks
    if "." not in domain_part:
        return False, "Email domain must contain a dot '.'."

    domain_segments = domain_part.split(".")
    # Check segment between @ and dot, and all intermediate segments
    for seg in domain_segments[:-1]:
        if len(seg) < 2:
            return False, f"Domain section '{seg}' must contain at least 2 characters (e.g. ab@cd.ef)."

    # Final top-level domain (after final dot)
    tld = domain_segments[-1]
    if len(tld) < 2:
        return False, "Domain suffix after the final dot must contain at least 2 characters (e.g. .com, .ef)."

    # Regex safety check
    pattern = r"^[A-Za-z0-9._%+-]{2,}@[A-Za-z0-9.-]{2,}\.[A-Za-z]{2,}$"
    if not re.match(pattern, email):
        return False, "Email contains unsupported characters. Please use a standard email address format."

    return True, ""

def validate_password_complexity(password: str, confirm_password: Optional[str] = None) -> Tuple[bool, str]:
    """
    Validates password complexity according to Milestone 1 Step 8:
    - Minimum 8 characters
    - At least one uppercase letter
    - At least one lowercase letter
    - At least one number
    - At least one special symbol
    - Confirm Password must exactly match Password
    """
    if not password:
        return False, "Password cannot be empty."

    if len(password) < 8:
        return False, "Password must be at least 8 characters long."

    if not re.search(r"[A-Z]", password):
        return False, "Password must contain at least one uppercase letter (A-Z)."

    if not re.search(r"[a-z]", password):
        return False, "Password must contain at least one lowercase letter (a-z)."

    if not re.search(r"[0-9]", password):
        return False, "Password must contain at least one numeric digit (0-9)."

    if not re.search(r"[!@#$%^&*(),.?\":{}|<>\-_=+[\]\\/`~;']", password):
        return False, "Password must contain at least one special symbol (!@#$%^&* etc.)."

    if confirm_password is not None:
        if password != confirm_password:
            return False, "Passwords do not match. Please verify Confirm Password."

    return True, ""

def hash_password(password: str) -> str:
    """Hashes a password using bcrypt."""
    salt = bcrypt.gensalt(rounds=12)
    return bcrypt.hashpw(password.encode("utf-8"), salt).decode("utf-8")

def verify_password(plain_password: str, hashed_password: str) -> bool:
    """Verifies a plain password against the stored bcrypt hash."""
    try:
        return bcrypt.checkpw(plain_password.encode("utf-8"), hashed_password.encode("utf-8"))
    except Exception:
        return False

def hash_security_answer(answer: str) -> str:
    """Normalizes and hashes security answer with SHA-256 for privacy."""
    normalized = answer.strip().lower()
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()

def verify_security_answer(user_input: str, stored_hash: str) -> bool:
    """Checks if the user's answer matches the stored hash."""
    return hash_security_answer(user_input) == stored_hash

def generate_jwt_token(username: str, email: str, role: str = "user", hours_valid: int = 2) -> str:
    """
    Generates a signed JWT session token with expiration.
    Payload includes username, email, role, issued at, and expiration.
    """
    now = datetime.datetime.now(datetime.timezone.utc)
    payload = {
        "sub": username,
        "email": email,
        "role": role,
        "iat": now,
        "exp": now + datetime.timedelta(hours=hours_valid)
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def verify_jwt_token(token: str) -> Tuple[bool, Optional[Dict[str, Any]], str]:
    """
    Decodes and validates a JWT session token.
    Returns: (is_valid, payload_dict, message)
    """
    if not token:
        return False, None, "No session token provided."

    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
        return True, payload, "Token is active and valid."
    except jwt.ExpiredSignatureError:
        return False, None, "Your session has expired. Please log in again."
    except jwt.InvalidTokenError:
        return False, None, "Invalid session token. Please log in again."
    except Exception as e:
        return False, None, f"Authentication error: {str(e)}"


In [ ]:
%%writefile app.py
import streamlit as st
import datetime
import hmac
from database import (
    init_db,
    create_user,
    get_user_by_identifier,
    get_user_by_username,
    get_user_by_email,
    update_password,
    get_all_users_for_admin,
    store_otp,
    verify_and_consume_otp
)
from auth import (
    SECURITY_QUESTIONS,
    validate_email_format,
    validate_password_complexity,
    hash_password,
    verify_password,
    hash_security_answer,
    verify_security_answer,
    generate_jwt_token,
    verify_jwt_token
)
from mailer import generate_otp, send_otp_email
from config import ADMIN_USERNAME, ADMIN_PASSWORD

# Ensure database tables exist
init_db()

# Page configuration
st.set_page_config(
    page_title="Infosys Springboard - Auth Module",
    page_icon="🛡️",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Custom CSS for rich styling and glassmorphism cards
st.markdown("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');
    
    html, body, [class*="css"] {
        font-family: 'Inter', sans-serif;
    }

    /* Main brand header */
    .brand-banner {
        background: linear-gradient(135deg, #002f6c 0%, #0070ad 50%, #00a4e4 100%);
        padding: 24px;
        border-radius: 14px;
        color: white;
        margin-bottom: 25px;
        box-shadow: 0 8px 24px rgba(0, 112, 173, 0.2);
    }
    .brand-title {
        font-size: 26px;
        font-weight: 700;
        margin: 0;
        letter-spacing: -0.5px;
    }
    .brand-subtitle {
        font-size: 14px;
        opacity: 0.9;
        margin-top: 5px;
        font-weight: 400;
    }

    /* Cards */
    .auth-card {
        background: #ffffff;
        border: 1px solid #e2e8f0;
        border-radius: 12px;
        padding: 25px;
        box-shadow: 0 4px 15px rgba(0, 0, 0, 0.05);
        margin-bottom: 20px;
    }

    /* Metric cards */
    .metric-card {
        background: #f8fafc;
        border-left: 4px solid #0070ad;
        padding: 16px;
        border-radius: 8px;
        margin-bottom: 12px;
        box-shadow: 0 2px 6px rgba(0,0,0,0.03);
    }

    /* Status badge */
    .status-badge {
        display: inline-block;
        padding: 4px 12px;
        font-size: 12px;
        font-weight: 600;
        border-radius: 20px;
        background-color: #e0f2fe;
        color: #0369a1;
    }
    .admin-badge {
        background-color: #fef3c7;
        color: #92400e;
    }
    
    /* Subtle divider */
    .section-divider {
        margin: 20px 0;
        border-top: 1px solid #e2e8f0;
    }
</style>
""", unsafe_allow_html=True)

# Session State Initialization
if "jwt_token" not in st.session_state:
    st.session_state["jwt_token"] = None

if "user" not in st.session_state:
    st.session_state["user"] = None

if "active_page" not in st.session_state:
    st.session_state["active_page"] = "Login"

if "otp_sent_email" not in st.session_state:
    st.session_state["otp_sent_email"] = None


def check_auth() -> bool:
    """Verifies currently stored JWT token in session state."""
    token = st.session_state.get("jwt_token")
    if not token:
        st.session_state["user"] = None
        return False
    is_valid, payload, _ = verify_jwt_token(token)
    if is_valid and payload:
        st.session_state["user"] = payload
        return True
    else:
        st.session_state["jwt_token"] = None
        st.session_state["user"] = None
        return False


def logout():
    """Clears user session state and redirects to Login."""
    st.session_state["jwt_token"] = None
    st.session_state["user"] = None
    st.session_state["otp_sent_email"] = None
    st.session_state["active_page"] = "Login"
    st.rerun()


# Global Header
st.markdown("""
<div class="brand-banner">
    <div class="brand-title">🛡️ Infosys Springboard Internship 7.0 (Batch 4)</div>
    <div class="brand-subtitle">Milestone 1 · Secure Authentication Module (Streamlit + JWT + SQLite + Gmail OTP)</div>
</div>
""", unsafe_allow_html=True)

is_logged_in = check_auth()
current_user = st.session_state.get("user")

# Sidebar Navigation
with st.sidebar:
    st.image("https://img.icons8.com/color/96/000000/security-shield-green.png", width=64)
    st.title("Navigation")

    if is_logged_in:
        role = current_user.get("role", "user")
        st.success(f"Signed in as **{current_user.get('sub')}**")
        st.markdown(f"<span class='status-badge {'admin-badge' if role == 'admin' else ''}'>Role: {role.upper()}</span>", unsafe_allow_html=True)
        st.markdown("<br>", unsafe_allow_html=True)
        
        if role == "admin":
            st.session_state["active_page"] = "Admin Dashboard"
            st.info("You have administrative access.")
        else:
            st.session_state["active_page"] = "User Dashboard"

        if st.button("🚪 Logout", use_container_width=True, type="primary"):
            logout()
    else:
        nav_options = ["Login", "Signup", "Forgot Password", "Admin Portal"]
        
        # Keep selected page synced
        current_index = 0
        if st.session_state["active_page"] in nav_options:
            current_index = nav_options.index(st.session_state["active_page"])
            
        selected_nav = st.radio(
            "Go to Page:",
            nav_options,
            index=current_index,
            key="sidebar_nav_radio"
        )
        if selected_nav != st.session_state["active_page"]:
            st.session_state["active_page"] = selected_nav
            st.rerun()

    st.markdown("---")
    st.caption("🔒 **Security & Session Details**")
    st.caption("• Algorithm: **HS256**")
    st.caption("• Token Lifetime: **2 Hours**")
    st.caption("• Password Storage: **bcrypt**")


# ==============================================================================
# 1. LOGIN PAGE (Step 3 & Step 7)
# ==============================================================================
if not is_logged_in and st.session_state["active_page"] == "Login":
    st.subheader("🔑 User Login")
    st.write("Please sign in using your registered username or email address.")

    col1, col2 = st.columns([2, 1])
    with col1:
        with st.form("login_form"):
            login_id = st.text_input("Username or Email", placeholder="e.g. john_doe or ab@cd.ef").strip()
            password = st.text_input("Password", type="password", placeholder="Enter your password").strip()
            submit_login = st.form_submit_button("Sign In", use_container_width=True, type="primary")

            if submit_login:
                # Step 8 Mandatory check
                if not login_id or not password:
                    st.error("⚠️ All fields are mandatory. Please provide both username/email and password.")
                else:
                    user = get_user_by_identifier(login_id)
                    # Step 3: Generic error on failure without revealing whether username or password was wrong
                    if not user or not verify_password(password, user["password_hash"]):
                        st.error("❌ Invalid username/email or password.")
                    else:
                        # Step 7: Issue fresh JWT token on Login
                        token = generate_jwt_token(user["username"], user["email"], role="user")
                        st.session_state["jwt_token"] = token
                        st.session_state["user"] = {
                            "sub": user["username"],
                            "email": user["email"],
                            "role": "user"
                        }
                        st.session_state["active_page"] = "User Dashboard"
                        st.success("✅ Login successful! Redirecting to dashboard...")
                        st.rerun()

    with col2:
        st.info("""
        **Help & Shortcuts:**
        - Need an account? Head over to **Signup**.
        - Forgot your password? Use **Forgot Password** to recover via Security Question or Gmail OTP.
        - Staff / Evaluator? Use **Admin Portal**.
        """)


# ==============================================================================
# 2. SIGNUP PAGE (Step 3 & Step 8)
# ==============================================================================
elif not is_logged_in and st.session_state["active_page"] == "Signup":
    st.subheader("📝 New User Registration")
    st.write("Create a secure profile for Infosys Springboard Milestone 1.")

    col1, col2 = st.columns([2, 1])
    with col1:
        with st.form("signup_form"):
            username = st.text_input("Username *", placeholder="Choose a unique username").strip()
            email = st.text_input("Email Address *", placeholder="e.g. ab@cd.ef").strip()
            
            pwd_col1, pwd_col2 = st.columns(2)
            with pwd_col1:
                password = st.text_input("Password *", type="password", placeholder="Min. 8 chars with uppercase, digits, symbols").strip()
            with pwd_col2:
                confirm_password = st.text_input("Confirm Password *", type="password", placeholder="Repeat exact password").strip()

            security_question = st.selectbox(
                "Security Question *",
                SECURITY_QUESTIONS,
                index=0
            )
            security_answer = st.text_input("Security Answer *", placeholder="Answer used for recovery").strip()

            submit_signup = st.form_submit_button("Register Account", use_container_width=True, type="primary")

            if submit_signup:
                # Step 8: Mandatory field check
                if not username or not email or not password or not confirm_password or not security_answer:
                    st.error("⚠️ All fields are mandatory. Please fill in all required inputs.")
                else:
                    # Step 8: Email format validation
                    is_email_valid, email_err = validate_email_format(email)
                    if not is_email_valid:
                        st.error(f"❌ {email_err}")
                    else:
                        # Step 8: Password complexity & match validation
                        is_pwd_valid, pwd_err = validate_password_complexity(password, confirm_password)
                        if not is_pwd_valid:
                            st.error(f"❌ {pwd_err}")
                        else:
                            # Hash password and security answer
                            pwd_hash = hash_password(password)
                            ans_hash = hash_security_answer(security_answer)

                            # Save to SQLite
                            success, msg = create_user(
                                username=username,
                                email=email,
                                password_hash=pwd_hash,
                                security_question=security_question,
                                security_answer_hash=ans_hash
                            )

                            if success:
                                st.success("🎉 Account created successfully! Please proceed to Login.")
                                st.session_state["active_page"] = "Login"
                            else:
                                st.error(f"❌ {msg}")

    with col2:
        st.markdown("""
        <div class="metric-card">
            <h4>Password Requirements</h4>
            <ul style="font-size: 13px; margin: 0; padding-left: 20px;">
                <li>At least 8 characters</li>
                <li>At least one uppercase letter (A-Z)</li>
                <li>At least one lowercase letter (a-z)</li>
                <li>At least one numeric digit (0-9)</li>
                <li>At least one special symbol (!@#$%^&*)</li>
            </ul>
        </div>
        <div class="metric-card">
            <h4>Email Format Rule</h4>
            <p style="font-size: 13px; margin: 0;">
                Must have ≥2 letters before '@', ≥2 letters between '@' and '.', and ≥2 letters after '.' (e.g. <code>ab@cd.ef</code>).
            </p>
        </div>
        """, unsafe_allow_html=True)


# ==============================================================================
# 3. FORGOT PASSWORD PAGE (Step 8 & Step 9)
# ==============================================================================
elif not is_logged_in and st.session_state["active_page"] == "Forgot Password":
    st.subheader("🔄 Password Recovery")
    st.write("Recover access to your account using your Security Question or Gmail OTP.")

    tab_sq, tab_otp = st.tabs(["🛡️ Route 1: Security Question", "📧 Route 2: Gmail OTP"])

    # --- Route 1: Security Question Route ---
    with tab_sq:
        st.markdown("#### Recover via Security Question")
        st.caption("Verify your stored security question and answer to set a new password.")

        sq_username = st.text_input("Enter your Username", key="sq_username_input").strip()

        if sq_username:
            user_data = get_user_by_username(sq_username)
            if not user_data:
                st.warning("⚠️ No account found with that username.")
            else:
                st.info(f"**Security Question:** {user_data['security_question']}")

                with st.form("sq_reset_form"):
                    sq_answer = st.text_input("Your Security Answer *", placeholder="Enter your answer").strip()
                    sq_new_pwd = st.text_input("New Password *", type="password", placeholder="Enter compliant new password").strip()
                    sq_confirm_pwd = st.text_input("Confirm New Password *", type="password", placeholder="Repeat new password").strip()

                    submit_sq_reset = st.form_submit_button("Reset Password", use_container_width=True, type="primary")

                    if submit_sq_reset:
                        if not sq_answer or not sq_new_pwd or not sq_confirm_pwd:
                            st.error("⚠️ All fields are mandatory.")
                        elif not verify_security_answer(sq_answer, user_data["security_answer_hash"]):
                            st.error("❌ Incorrect security answer. Please try again.")
                        else:
                            is_pwd_valid, pwd_err = validate_password_complexity(sq_new_pwd, sq_confirm_pwd)
                            if not is_pwd_valid:
                                st.error(f"❌ {pwd_err}")
                            else:
                                new_hash = hash_password(sq_new_pwd)
                                if update_password(user_data["username"], new_hash):
                                    st.success("✅ Password updated successfully! Redirecting to Login...")
                                    st.session_state["active_page"] = "Login"
                                    st.rerun()
                                else:
                                    st.error("❌ Failed to update password. Please try again.")

    # --- Route 2: Gmail OTP Route ---
    with tab_otp:
        st.markdown("#### Recover via Gmail OTP")
        st.caption("We will send a 6-digit one-time code to your registered email address.")

        otp_col1, otp_col2 = st.columns([3, 1])
        with otp_col1:
            target_email = st.text_input(
                "Enter Registered Email Address",
                value=st.session_state.get("otp_sent_email") or "",
                placeholder="e.g. ab@cd.ef"
            ).strip()

        with otp_col2:
            st.write("")
            st.write("")
            send_btn = st.button("📨 Send OTP", use_container_width=True)

        if send_btn:
            if not target_email:
                st.error("⚠️ Please enter your registered email address first.")
            else:
                target_user = get_user_by_email(target_email)
                if not target_user:
                    st.error("❌ No account found with this email address.")
                else:
                    otp_code = generate_otp()
                    with st.spinner("Connecting to Gmail SMTP and sending verification OTP..."):
                        success, mail_msg = send_otp_email(target_email, otp_code)
                    if success:
                        store_otp(target_email, otp_code, validity_minutes=10)
                        st.session_state["otp_sent_email"] = target_email
                        st.success(f"✅ One-Time Password successfully sent to {target_email}! Please check your Gmail inbox and spam folder.")
                    else:
                        st.session_state["otp_sent_email"] = None
                        st.error(f"❌ OTP Delivery Failed: {mail_msg}")

        # Reset Form with OTP
        if st.session_state.get("otp_sent_email"):
            st.markdown("---")
            st.markdown(f"**Verification for:** `{st.session_state['otp_sent_email']}`")
            with st.form("otp_reset_form"):
                user_otp = st.text_input("Enter 6-Digit OTP *", max_chars=6, placeholder="e.g. 123456").strip()
                otp_new_pwd = st.text_input("New Password *", type="password", placeholder="Enter compliant new password").strip()
                otp_confirm_pwd = st.text_input("Confirm New Password *", type="password", placeholder="Repeat new password").strip()

                submit_otp_reset = st.form_submit_button("Verify OTP & Update Password", use_container_width=True, type="primary")

                if submit_otp_reset:
                    if not user_otp or not otp_new_pwd or not otp_confirm_pwd:
                        st.error("⚠️ All fields are mandatory.")
                    else:
                        is_valid_otp, otp_err = verify_and_consume_otp(st.session_state["otp_sent_email"], user_otp)
                        if not is_valid_otp:
                            st.error(f"❌ {otp_err}")
                        else:
                            is_pwd_valid, pwd_err = validate_password_complexity(otp_new_pwd, otp_confirm_pwd)
                            if not is_pwd_valid:
                                st.error(f"❌ {pwd_err}")
                            else:
                                user_obj = get_user_by_email(st.session_state["otp_sent_email"])
                                if user_obj:
                                    new_hash = hash_password(otp_new_pwd)
                                    update_password(user_obj["username"], new_hash)
                                    st.session_state["otp_sent_email"] = None
                                    st.success("🎉 Password successfully reset! Redirecting to Login...")
                                    st.session_state["active_page"] = "Login"
                                    st.rerun()


# ==============================================================================
# 4. ADMIN LOGIN PORTAL (Step 11)
# ==============================================================================
elif not is_logged_in and st.session_state["active_page"] == "Admin Portal":
    st.subheader("🛡️ Administrator Access")
    st.write("Separate authentication portal for system administrators and evaluators.")

    col1, col2 = st.columns([2, 1])
    with col1:
        with st.form("admin_login_form"):
            admin_user = st.text_input("Admin Username", placeholder="e.g. admin").strip()
            admin_pwd = st.text_input("Admin Password", type="password", placeholder="Enter admin password").strip()
            submit_admin = st.form_submit_button("Authenticate as Admin", use_container_width=True, type="primary")

            if submit_admin:
                if not admin_user or not admin_pwd:
                    st.error("⚠️ Both admin username and password are mandatory.")
                # Verify securely against configured admin credentials (from Colab secrets or config)
                is_user_match = hmac.compare_digest(admin_user.strip().lower(), ADMIN_USERNAME.strip().lower())
                is_pwd_match = hmac.compare_digest(admin_pwd.strip(), ADMIN_PASSWORD.strip())
                if is_user_match and is_pwd_match:
                    # Issue admin JWT token
                    token = generate_jwt_token(admin_user, "admin@infosys.springboard.internal", role="admin")
                    st.session_state["jwt_token"] = token
                    st.session_state["user"] = {
                        "sub": admin_user,
                        "email": "admin@infosys.springboard.internal",
                        "role": "admin"
                    }
                    st.session_state["active_page"] = "Admin Dashboard"
                    st.success("✅ Admin authentication successful! Redirecting...")
                    st.rerun()
                else:
                    st.error("❌ Invalid admin credentials.")

    with col2:
        st.warning("""
        **Notice:**
        This portal is strictly reserved for authorized administrators. All activities are recorded for audit compliance.
        """)


# ==============================================================================
# 5. PROTECTED USER DASHBOARD (Step 10)
# ==============================================================================
elif is_logged_in and current_user and current_user.get("role") == "user":
    st.subheader(f"👋 Welcome to your Dashboard, {current_user.get('sub')}!")
    st.caption("You are authenticated via a verified, cryptographic JWT session token.")

    # Top summary metrics
    mcol1, mcol2, mcol3 = st.columns(3)
    with mcol1:
        st.markdown("""
        <div class="metric-card">
            <span style="font-size: 12px; color: #64748b; font-weight: 600;">ACTIVE USER</span>
            <h3 style="margin: 4px 0; color: #0070ad;">{}</h3>
            <span style="font-size: 12px; color: #0284c7;">Role: Standard User</span>
        </div>
        """.format(current_user.get("sub")), unsafe_allow_html=True)

    with mcol2:
        st.markdown("""
        <div class="metric-card">
            <span style="font-size: 12px; color: #64748b; font-weight: 600;">REGISTERED EMAIL</span>
            <h3 style="margin: 4px 0; color: #0f172a; font-size: 18px;">{}</h3>
            <span style="font-size: 12px; color: #16a34a;">Status: Verified</span>
        </div>
        """.format(current_user.get("email")), unsafe_allow_html=True)

    with mcol3:
        exp_timestamp = current_user.get("exp")
        exp_str = datetime.datetime.fromtimestamp(exp_timestamp, datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC") if exp_timestamp else "N/A"
        st.markdown("""
        <div class="metric-card">
            <span style="font-size: 12px; color: #64748b; font-weight: 600;">JWT TOKEN EXPIRATION</span>
            <h3 style="margin: 4px 0; color: #b45309; font-size: 18px;">{}</h3>
            <span style="font-size: 12px; color: #64748b;">Algorithm: HS256</span>
        </div>
        """.format(exp_str), unsafe_allow_html=True)

    st.markdown("### 📜 Verified Session Details")
    with st.expander("🔍 Inspect Decoded JWT Token Payload (Debug & Verification)", expanded=True):
        st.json(current_user)

    st.markdown("### 🚀 Internship Milestone 1 Progress Checklist")
    c1, c2 = st.columns(2)
    with c1:
        st.checkbox("Step 1: Repository & Milestone1 Folder Created", value=True, disabled=True)
        st.checkbox("Step 2: Comprehensive README.md Documented", value=True, disabled=True)
        st.checkbox("Step 3: Multi-Page Streamlit App Implemented", value=True, disabled=True)
        st.checkbox("Step 4: ngrok Public Tunnel Integration Configured", value=True, disabled=True)
        st.checkbox("Step 5: Gmail App Password / SMTP Configured", value=True, disabled=True)
        st.checkbox("Step 6: Colab Secrets Handled Without Hardcoding", value=True, disabled=True)
    with c2:
        st.checkbox("Step 7: Cryptographic JWT Session Management", value=True, disabled=True)
        st.checkbox("Step 8: Mandatory, Email (ab@cd.ef) & Password Validation", value=True, disabled=True)
        st.checkbox("Step 9: Forgot Password (Security Question + Gmail OTP)", value=True, disabled=True)
        st.checkbox("Step 10: Protected User Dashboard with Logout", value=True, disabled=True)
        st.checkbox("Step 11: Dedicated Admin Portal (Never Displays Passwords)", value=True, disabled=True)
        st.checkbox("Step 12: Clean Colab Notebook (.ipynb) & Screenshots", value=True, disabled=True)

    st.markdown("---")
    if st.button("🚪 Logout from User Session", type="primary"):
        logout()


# ==============================================================================
# 6. ADMIN DASHBOARD (Step 11)
# ==============================================================================
elif is_logged_in and current_user and current_user.get("role") == "admin":
    st.subheader("🛡️ Infosys Springboard — Admin Control Center")
    st.caption("Administrative oversight, user audit logs, and security monitoring.")

    all_users = get_all_users_for_admin()

    mcol1, mcol2, mcol3 = st.columns(3)
    with mcol1:
        st.metric(label="Total Registered Users", value=len(all_users))
    with mcol2:
        st.metric(label="Active Auth Engine", value="JWT (HS256)")
    with mcol3:
        st.metric(label="Password Hash Standard", value="bcrypt (12 rounds)")

    st.markdown("---")
    st.markdown("### 👥 Registered Users Directory")
    st.caption("⚠️ **Compliance Notice (Step 11):** Passwords and password hashes are strictly redacted from administrative views.")

    # Search filter
    search_query = st.text_input("🔍 Search users by username or email:", placeholder="Type to filter...").strip().lower()

    filtered_users = all_users
    if search_query:
        filtered_users = [
            u for u in all_users
            if search_query in u["username"].lower() or search_query in u["email"].lower()
        ]

    if filtered_users:
        # Display as clean table
        table_data = []
        for idx, u in enumerate(filtered_users, start=1):
            table_data.append({
                "User ID": u["id"],
                "Username": u["username"],
                "Email": u["email"],
                "Security Question Configured": u["security_question"],
                "Registration Timestamp": u["created_at"],
                "Account Status": "Active ✅"
            })
        st.dataframe(table_data, use_container_width=True)
    else:
        st.info("No registered users found matching the search criteria.")

    st.markdown("---")
    if st.button("🚪 Logout from Admin Portal", type="primary"):
        logout()


### 🚀 STEP 3: Run Streamlit and Launch ngrok Public Tunnel (Step 4 & Step 12)
Run the cell below to start the Streamlit server on port `8501` in the background and establish a secure HTTPS tunnel via `pyngrok`.

In [ ]:
import subprocess
import time
from pyngrok import ngrok
import os
from pathlib import Path

# Terminate existing Streamlit and ngrok tunnels
!pkill -f streamlit || true
ngrok.kill()

# 1. Sync latest secrets from Colab userdata into environment
try:
    from google.colab import userdata
    # Re-apply ngrok authtoken
    for k in ['NGROK_AUTHTOKEN', 'ngrok_authtoken', 'NGROK_AUTH_TOKEN', 'AUTHTOKEN']:
        try:
            ngrok_token = userdata.get(k)
            if ngrok_token:
                ngrok.set_auth_token(str(ngrok_token).strip())
                os.environ['NGROK_AUTHTOKEN'] = str(ngrok_token).strip()
                print('✅ ngrok authtoken applied from Colab Secrets.')
                break
        except Exception:
            pass
    # Re-sync Email and JWT secrets into os.environ
    for primary, aliases in [
        ('EMAIL_ADDRESS', ['email_address', 'GMAIL_ADDRESS', 'EMAIL']),
        ('EMAIL_PASSWORD', ['email_password', 'GMAIL_APP_PASSWORD', 'APP_PASSWORD']),
        ('JWT_SECRET', ['jwt_secret', 'SECRET_KEY']),
        ('ADMIN_USERNAME', ['admin_username']),
        ('ADMIN_PASSWORD', ['admin_password'])
    ]:
        for k in [primary] + aliases:
            try:
                val = userdata.get(k)
                if val:
                    os.environ[primary] = str(val).strip()
                    break
            except Exception:
                pass
except Exception as e:
    print(f'ℹ️ Colab secrets sync note: {e}')

# 2. Launch Streamlit app in background with populated environment
process = subprocess.Popen(
    ['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.headless', 'true'],
    env=os.environ.copy(),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(5)  # Wait for Streamlit server to boot

# 3. Connect ngrok tunnel to port 8501
public_url = ngrok.connect(8501)
print('=' * 65)
print(f'🎉 STREAMLIT APP IS PUBLICLY LIVE AT:\n👉 {public_url}')
print('=' * 65)
print('Open the link above in your web browser to test all authentication features.')


### 🧪 Testing & Verification Guide
1. **New User Registration (Signup)**:
   - Click **Signup** in the sidebar.
   - Test email rule (`ab@cd.ef`) and password rules (minimum 8 characters, uppercase, lowercase, digit, special character).
   - Choose a security question and enter an answer.
2. **User Login & JWT Verification**:
   - Enter username/email and password.
   - On success, redirected to **User Dashboard** with JWT payload and token expiration timer.
3. **Password Recovery**:
   - **Route 1 (Security Question)**: Enter username, answer question, update password.
   - **Route 2 (Gmail OTP)**: Enter registered email, click **Send OTP**, enter 6-digit code received in Gmail, set new password.
4. **Admin Portal**:
   - Click **Admin Portal** in the sidebar.
   - Enter admin credentials (configured in Colab Secrets or default).
   - View user count, search table of registered users (**passwords and hashes strictly redacted**).

> ⚠️ **Submission Note (Step 12):** Before submitting, ensure all cell outputs are cleared (Edit -> Clear all outputs) and no personal tokens or credentials remain visible.